# Lab | RAG from Scratch
## A privacy assistant for the GDPR

The bank from this week's labs has a data protection team, and it gets the same questions every day, from customers and from colleagues: *Can I ask you to delete my data? How long do we have to report a data breach? Do we need consent for this?* The answers are all in one document, the **General Data Protection Regulation (GDPR)**, Europe's data protection law, but it is 88 pages long.

Your job is to build a first version of a **privacy assistant**: it answers questions from the GDPR, cites the passages it used so the team can check them, and says so when the GDPR does not answer the question. Then you **evaluate** it on 10 questions with known answers, to decide whether it is good enough to show the team.

**This lab practises** retrieval-augmented generation. You will use, from today's **RAG from Scratch** lesson: loading a PDF, chunking with overlap and metadata, embeddings with MiniLM, a Chroma collection, retrieval with metadata filters, a grounded prompt with citations and refusals, and the hit rate and citation checks.

**By the end, you can:**

1. Implement a RAG pipeline (chunk, embed, store, retrieve, generate) over a new document.
2. Evaluate it with a test set, and explain whether each failure comes from retrieval or from generation.

Each challenge ends with a **checkpoint** that tells you what you should see. Retrieval gives the same results every time; the LLM's answers change a little from run to run. **The whole lab costs about one US cent** with the model below; the last cell shows your total.

**The document** is `data/gdpr.pdf`, [Regulation (EU) 2016/679](https://eur-lex.europa.eu/eli/reg/2016/679/oj) as published in the Official Journal, from EUR-Lex (© European Union, https://eur-lex.europa.eu, reused under [Commission Decision 2011/833/EU](https://eur-lex.europa.eu/eli/dec/2011/833/oj)). Nothing your assistant says is legal advice.

## Setup

You need `pip install pypdf sentence-transformers chromadb openai python-dotenv`. As in yesterday's lab, create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...`. The repository's `.gitignore` already lists `.env`: **never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries, the OpenAI client, a `cost` function and a `track` function that logs the tokens of every call, as yesterday. It also gives you `load_pdf`, the loader from the lesson, with the noise pattern changed for this document: the GDPR prints a footer such as `4.5.2016 L 119/41 Official Journal of the European Union EN` on every page.

In [1]:
pip install pypdf sentence-transformers chromadb openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import logging
import warnings
from collections import Counter

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 90)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []

def cost(input_tokens, output_tokens):
    price_in, price_out = PRICES[MODEL]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

NOISE = re.compile(r"L 119/\d+ Official Journal of the European Union")

def load_pdf(path):
    """Return one dictionary per page, {"page": number, "text": text}, without the page footers."""
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        lines = [line.strip() for line in page.extract_text().split("\n")]
        lines = [line for line in lines if line and not NOISE.search(line)]
        pages.append({"page": number, "text": "\n".join(lines)})
    return pages

Key found: True


**Checkpoint:** `Key found: True`, and no error.

## Challenge 1: load and chunk the GDPR

1. Load `../data/gdpr.pdf` with `load_pdf`, and print the number of pages and the total number of words.
2. Find the page where **Article 17** (*Right to erasure*) starts, and print the first 600 characters of that page. Look at the text: is it clean enough to use?
3. Write `chunk_document(pages, size=150, overlap=30)`. It returns a list of chunks, each a dictionary with an `id` (`"gdpr-0"`, `"gdpr-1"`, ...), the `text`, the `page` where the chunk starts and its `section`: `"Recitals"` before Article 1, then `"Article 1"`, `"Article 2"`, and so on (the section most of the chunk's words come from). In the GDPR, every article starts with a line such as `Article 17`.
4. Chunk the document and print the number of chunks and the share that are recitals.
5. Load the embedding model with `embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`, count the tokens of every chunk with `embedder.tokenizer`, and print the median and the share of chunks over the model's 256-token limit.

In [3]:
# 1. Load
pages = load_pdf('../data/gdpr.pdf')
words_per_page = [len(p['text'].split()) for p in pages]
print(f'Number of pages: {len(pages)} | Number of words: {sum(words_per_page)}')


# 2. The page of Article 17
for page in pages:
    for line in page['text'].split('\n'):
        if line == 'Article 17':
            print(f'Article 17 starts on page: {page['page']}')
            print(page["text"][:600])

# 3. Chunking function
HEADING = re.compile(r"^(Article \d+)$")

def chunk_document(pages, size=150, overlap=30):
    words =[]
    section = 'Recitals'
    for page in pages:
        for line in page['text'].split('\n'):
            match = HEADING.match(line)
            if match:
                section = match.group(1)
            for word in line.split():
                words.append((word, page['page'], section))
    
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        chunks.append({
            "id": f"gdpr-{len(chunks)}",
            "text": " ".join(word for word, _, _ in piece),
            "page": piece[0][1],
            "section": Counter(section for _, _, section in piece).most_common(1)[0][0],
        })
        if start + size >= len(words):
            break
    return chunks


# 4. Chunks and recitals
chunks = chunk_document(pages)
sections = pd.Series([c["section"] for c in chunks])
print(len(chunks), "chunks")
print(f"Recitals: {(sections == 'Recitals').mean():.0%}")

# 5. Tokens per chunk
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Maximum input:", embedder.max_seq_length, "tokens")

rows = []
for size in [50, 150, 400]:
    sample = chunk_document(pages, size=size, overlap=size // 5)
    tokens = [len(embedder.tokenizer(c["text"])["input_ids"]) for c in sample]
    rows.append({"words per chunk": size, "chunks": len(sample), "median tokens": int(np.median(tokens)),
                 "share over 256 tokens": f"{np.mean(np.array(tokens) > 256):.0%}"})
pd.DataFrame(rows)

Number of pages: 88 | Number of words: 54932
Article 17 starts on page: 43
Article 15
Right of access by the data subject
1. The data subject shall have the right to obtain from the controller confirmation as to whether or not personal data
concerning him or her are being processed, and, where that is the case, access to the personal data and the following
information:
(a)  the purposes of the processing;
(b)  the categories of personal data concerned;
(c)  the recipients or categories of recipient to whom the personal data have been or will be disclosed, in particular
recipients in third countries or international organisations;
(d)  where possible, the envisaged pe
458 chunks
Recitals: 45%
Maximum input: 256 tokens


,words per chunk,chunks,median tokens,share over 256 tokens
0,50,1374,59,0%
1,150,458,174,0%
2,400,172,463,99%


**Checkpoint:** 88 pages and about 55,000 words. Article 17 starts on page 43, and the text is clean: the heading, the title *Right to erasure ('right to be forgotten')* and the numbered paragraphs, with no footer. About 458 chunks, of which about 45% are recitals (the GDPR has 173 recitals for 99 articles). A median of about 174 tokens per chunk and none over 256: 150-word chunks fit MiniLM.

## Challenge 2: index the chunks

1. Embed the text of every chunk with `embedder`, normalised, and print the shape of the result.
2. Create a Chroma collection called `"gdpr"` (in memory, with `chromadb.Client()`) that compares vectors with cosine distance, and add the ids, texts, embeddings and metadata (`page` and `section`) of all chunks. Print `collection.count()`.

In [4]:
# 1. Embeddings
embeddings = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print(embeddings.shape)


# 2. Collection
chroma = chromadb.Client()
def make_collection(name, chunks, embeddings):
    collection = chroma.create_collection(name, configuration={'hnsw':{'space':'cosine'}})
    collection.add(
        ids=[c['id'] for c in chunks],
        documents=[c['text'] for c in chunks],
        embeddings=embeddings,
        metadatas=[{"page": c['page'], "section": c['section']} for c in chunks]
    )
    return collection

collection = make_collection("gdpr", chunks, embeddings)
print("Chunks in the collection:", collection.count())

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

(458, 384)
Chunks in the collection: 458


**Checkpoint:** `(458, 384)` and `458 chunks in the collection` (the same number as your chunks, whatever it is).

## Challenge 3: retrieve

1. Write `retrieve(question, k=5, where=None)`: it embeds the question with the same model, queries the collection, and returns a DataFrame with the columns `id`, `section`, `page`, `distance` and `text`.
2. Define `RULES_ONLY`, a `where` filter that excludes the recitals.
3. Retrieve the top 5 chunks for *Can I ask a company to delete my data?*, first without a filter and then with `RULES_ONLY`.

In [5]:
# 1. Retrieval function
def retrive(question, k=5, where=None):
    query_embedding = embedder.encode([question], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        where=where
    )
    return pd.DataFrame({
        'id': results['ids'][0],
        'section': [m['section'] for m in results['metadatas'][0]],
        'page': [m['page'] for m in results['metadatas'][0]],
        'distance': np.round(results['distances'][0], 3),
        'text': results['documents'][0]
    })


# 2. Filter
RULES_ONLY = {"section": {"$ne": "Recitals"}}


# 3. Without and with the filter
without_filter = retrive("Can I ask a company to delete my data?")
with_filter = retrive("Can I ask a company to delete my data?", where=RULES_ONLY)

In [6]:
display("Without filter:", without_filter)
display("With filter:", with_filter)

'Without filter:'

,id,section,page,distance,text
0,gdpr-255,Article 15,43,0.508,right to request from the controller rectification or erasure of personal data or rest...
1,gdpr-186,Recitals,29,0.512,"data in the employment context, in particular for the conditions under which personal ..."
2,gdpr-76,Recitals,12,0.514,in particular in the context of online services and online identifiers. A controller s...
3,gdpr-77,Recitals,12,0.533,has withdrawn his or her consent or objects to the processing of personal data concern...
4,gdpr-78,Recitals,13,0.536,"freedom of expression and information, for compliance with a legal obligation, for the..."


'With filter:'

,id,section,page,distance,text
0,gdpr-255,Article 15,43,0.508,right to request from the controller rectification or erasure of personal data or rest...
1,gdpr-246,Article 13,41,0.538,"Article 6(1) or point (a) of Article 9(2), the existence of the right to withdraw cons..."
2,gdpr-257,Article 17,43,0.541,right to obtain from the controller without undue delay the rectification of inaccurat...
3,gdpr-220,Article 5,35,0.542,"are processed, are erased or rectified without delay (‘accuracy’); (1) Directive (EU) ..."
4,gdpr-444,Article 88,84,0.548,of the rights and freedoms in respect of the processing of employees' personal data in...


**Checkpoint:** Without the filter, the first chunk comes from Article 15 and the other four are recitals. With `RULES_ONLY`, all five are articles, and Article 17 (page 43) is third.

**Question:** which article answers the question, and in which position does it appear with and without the filter? Why do the recitals compete with the articles?

Article 17 answers the question, which does not appear at all without the filter and it appears in third place with filter. The recitals compete because they explain the same rules with similar words and they make up 45% of the chunks.


## Challenge 4: grounded answers

1. Write the system prompt of the privacy assistant. It must answer **only** from the excerpts, cite the excerpt ids in square brackets after each statement (for example `[gdpr-12]`), reply exactly `I don't know based on the GDPR.` and nothing else when the excerpts do not contain the answer, and answer in 2 to 4 sentences of plain English.
2. Write `answer(question, k=5, where=RULES_ONLY)`. It retrieves the chunks, builds the context (each chunk labelled with its id, section and page, as in the lesson), calls the model with the system prompt and a user message containing the excerpts and the question, tracks the call as `"answers"`, and returns a dictionary with the `answer`, the `cited` ids (find them in the answer with a regular expression), the `retrieved` ids and their `sections`.
3. Print the answers to these three questions, each with the sections it cited:
   - *How long does a company have to report a data breach to the authority?*
   - *Can I get a copy of the personal data a company holds about me?*
   - *What is the VAT rate for online services in Spain?*

In [7]:
# 1. System prompt
SYSTEM = """
            You answer questions about the EU AI Act using ONLY the excerpts provided.
            Rules:
            - Base every statement on the excerpts. Do not use outside knowledge.
            - After each statement, cite the excerpt id(s) in square brackets, e.g. [gdpr-12].
            - If the excerpts do not contain the answer, reply exactly "I don't know based on the GDPR." and nothing else.
            - Answer in 2-4 sentences of plain English.
         """

# 2. Answer function
def build_context(hits):
    return "\n\n".join(f"[{row.id}] ({row.section}, page {row.page})" for row in hits.itertuples())

def answer(question, k=5, where=RULES_ONLY):
    hits = retrive(question, k=k, where=where)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Excerpts:\n{build_context(hits)}\n\nQuestion: {question}"},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    track(response, "answer")
    answer = response.choices[0].message.content
    cited = re.findall(r"gdpr-\d+", answer)
    answers = {
        "answer": answer, 
        "cited": sorted(set(cited)), 
        "retrieved": list(hits["id"]),
        "sections": list(hits["section"])
    }
    return answers

# 3. Three questions
section_of = {c["id"]: c["section"] for c in chunks}

questions = [
    "How long does a company have to report a data breach to the authority?",
    "Can I get a copy of the personal data a company holds about me?",
    "What is the VAT rate for online services in Spain?"
]

for question in questions:
    result = answer(question)
    print(question, "\n ->", result["answer"])
    print("Cited sections:", [section_of[c] for c in result["cited"]], "\n")

How long does a company have to report a data breach to the authority? 
 -> I don't know based on the GDPR.
Cited sections: [] 

Can I get a copy of the personal data a company holds about me? 
 -> Yes. The excerpts indicate you can request a copy of the personal data a company holds about you, as described under the relevant article on access to personal data. [gdpr-255] [gdpr-256]
Cited sections: ['Article 15', 'Article 15'] 

What is the VAT rate for online services in Spain? 
 -> I don't know based on the GDPR.
Cited sections: [] 



In [8]:
retrive("How long does a company have to report a data breach to the authority?", where=RULES_ONLY)

,id,section,page,distance,text
0,gdpr-295,Article 33,52,0.451,approximate number of data subjects concerned and the categories and approximate numbe...
1,gdpr-296,Article 34,52,0.483,"relating to the personal data breach, its effects and the remedial action taken. That ..."
2,gdpr-294,Article 33,52,0.486,"controller shall without undue delay and, where feasible, not later than 72 hours afte..."
3,gdpr-316,Article 40,57,0.495,data breaches to supervisory authorities and the communication of such personal data b...
4,gdpr-293,Article 32,52,0.497,approved code of conduct as referred to in Article 40 or an approved certification mec...


**Checkpoint:** The breach answer says *without undue delay* and, where feasible, *not later than 72 hours* after the company becomes aware of the breach, citing a chunk of Article 33. The copy answer says yes, citing Article 15 (and mentions that further copies can cost a reasonable fee). The VAT question gets `I don't know based on the GDPR.` (sometimes followed by chunk ids, even though the rule says *nothing else*). Your wording will differ.

**Question:** open `data/gdpr.pdf` at the page of one chunk your assistant cited, and compare. Is the answer faithful to the text? Did it leave out anything important?

On page 52, Article 33 states that the controller must notify a breach to the supervisory authority without undue delay and, where feasible, within 72 hours of becoming aware of it, and the cited chunk gdpr-294 contains that sentence. The assistant’s answer was not faithful. On the first run, it claimed the excerpts did not say how long a company has, and on each consequent run it returned the fallback "I don't know based on the GDPR", so it left out the most important information even though it was right there in the context.

## Challenge 5: evaluate the assistant

The data protection team wrote 10 questions and, for each one, the article that answers it. Use them to decide whether the assistant is ready to show.

1. Write `hit_rate(k=5, where=None)`: the share of test questions for which at least one of the top k chunks comes from the expected article. Print a table with the hit rate for `k` = 3 and 5, without a filter and with `RULES_ONLY`, and the questions missed in each case.
2. Run `answer` on the 10 questions and build a DataFrame with one row per question and four `True`/`False` columns: `right article retrieved`, `cites` (at least one id), `cites only what it saw` (every cited id was retrieved) and `cites the right article`. Print the mean of each column and show the rows that failed any check.
3. Run `answer` on the two out-of-scope questions in `OUT_OF_SCOPE` and count how many were refused.

In [9]:
TEST_SET = pd.DataFrame([
    ("Can I ask a company to delete my data?", "Article 17"),
    ("How long does a company have to tell the authority about a data breach?", "Article 33"),
    ("What are the legal grounds for processing personal data?", "Article 6"),
    ("How much can a company be fined under the GDPR?", "Article 83"),
    ("Can I get a copy of the personal data a company holds about me?", "Article 15"),
    ("At what age can a child consent to online services on their own?", "Article 8"),
    ("When does a company need a data protection officer?", "Article 37"),
    ("Can I move my data from one service provider to another?", "Article 20"),
    ("What counts as valid consent?", "Article 7"),
    ("Can a decision about me be made only by an algorithm?", "Article 22"),
], columns=["question", "expected"])
OUT_OF_SCOPE = ["What is the VAT rate for online services in Spain?",
                "How many vacation days do employees get in France?"]

# 1. Hit rate
def hit_rate(k=5, where=None):
    queries = embedder.encode(list(TEST_SET["question"]), normalize_embeddings=True)
    result = collection.query(query_embeddings=queries, n_results=k, where=where)
    hits = [expected in [m["section"] for m in metas]
            for expected, metas in zip(TEST_SET["expected"], result["metadatas"])]
    return np.mean(hits), [q for q, hit in zip(TEST_SET["question"], hits) if not hit]

rows = []
for k in [3, 5]:
    for name, where in [("none", None), ("rules only", RULES_ONLY)]:
        rate, missed = hit_rate(k=k, where=where)
        rows.append({"k": k, "filter": name, "hit rate": round(rate, 2), "missed": missed})
pd.DataFrame(rows)


,k,filter,hit rate,missed
0,3,none,0.5,"[Can I ask a company to delete my data?, How long does a company have to tell the auth..."
1,3,rules only,0.8,"[What are the legal grounds for processing personal data?, Can I move my data from one..."
2,5,none,0.6,"[Can I ask a company to delete my data?, What are the legal grounds for processing per..."
3,5,rules only,0.8,"[What are the legal grounds for processing personal data?, Can I move my data from one..."


In [10]:

# 2. Answer checks
section_of = {c["id"]: c["section"] for c in chunks}

rows = []
for question, expected in TEST_SET.itertuples(index=False):
    result = answer(question)
    rows.append({
        "question": question,
        "right article retrieved": expected in result["sections"],
        "cites": len(result["cited"]) > 0,
        "cites only what it saw": set(result["cited"]) <= set(result["retrieved"]),
        "cites the right article": expected in [section_of[c] for c in result["cited"]],
    })
checks = pd.DataFrame(rows)
print(checks.drop(columns="question").mean().round(2))
checks[~checks.drop(columns="question").all(axis=1)]


right article retrieved    0.8
cites                      0.2
cites only what it saw     1.0
cites the right article    0.2
dtype: float64


,question,right article retrieved,cites,cites only what it saw,cites the right article
1,How long does a company have to tell the authority about a data breach?,True,False,True,False
2,What are the legal grounds for processing personal data?,False,False,True,False
3,How much can a company be fined under the GDPR?,True,False,True,False
5,At what age can a child consent to online services on their own?,True,False,True,False
6,When does a company need a data protection officer?,True,False,True,False
7,Can I move my data from one service provider to another?,False,False,True,False
8,What counts as valid consent?,True,False,True,False
9,Can a decision about me be made only by an algorithm?,True,False,True,False


In [11]:

# 3. Out-of-scope questions
refused = 0
for question in OUT_OF_SCOPE:
    text = answer(question)["answer"]
    print(question, "\n ->", text, "\n")
    if "I don't know based on the GDPR" in text:
        refused += 1
print("Refused:", refused, "of", len(OUT_OF_SCOPE))

What is the VAT rate for online services in Spain? 
 -> I don't know based on the GDPR. 

How many vacation days do employees get in France? 
 -> I don't know based on the GDPR. 

Refused: 2 of 2


**Checkpoint:** Hit rates of 0.5 and 0.8 at k = 3, and 0.6 and 0.8 at k = 5 (without and with the filter). With the filter, the two misses are the legal grounds (Article 6) and data portability (Article 20). In the answer checks, about 0.8 of the answers cite the right article and none cites an id it did not see, and the failures are the same two questions, both with `right article retrieved` = `False`. Both out-of-scope questions are refused. The retrieval numbers should match exactly; the answers can differ a little.

**Question:** for each failure, say whether it comes from retrieval or from generation, and why. Then decide: would you show this assistant to the data protection team? What would you change first, and what would you tell them about how to use it?

Two failures come from retrieval: the legal grounds (Article 6) and portability (Article 20) questions use everyday words (“legal grounds”, “move my data”) that differ from the law’s wording, so the right article was never retrieved. 
The other six come from generation: the right article was retrieved, but the model did not cite it, applying “use ONLY the excerpts” too strictly when the question’s wording differs from the law’s, as with the 72-hour deadline in Challenge 4. 
I would not show the assistant to the team yet: it is honest (no invented citations, both out-of-scope questions refused), but it only answered 2 of 10 questions correctly. I would first try a stronger model or a better prompt, then fix retrieval and advise the team to treat it as a research aid, always check the cited article, and not take “I don’t know” as proof that the GDPR has no answer.

## What did it cost?

Run the cell below to see the tokens and the cost of the lab.

In [12]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

Total: 15 calls, $0.0010


,input,output,calls,cost ($)
step,,,,
answer,3012,284,15,0.00096


**Checkpoint:** 15 calls and about half a US cent. Indexing with MiniLM was free; every answer costs one call of about 1,100 input tokens.

## Extra: tune the chunk size, or use your own document

Choose one:

+ **Chunk size.** Build one collection per chunk size (100, 150, 200 and 400 words, with an overlap of a fifth of the size) and compare their hit rate at k = 5 with `RULES_ONLY`. Which size works best on the GDPR, and is it the same as for the AI Act in the lesson? Be careful: with only 10 questions, a difference of one question is 0.1.
+ **Your own document.** Index a PDF you are allowed to use (your company's public privacy policy, a manual, a public report), write five questions with the page that answers them, and measure the hit rate. What did you have to change in `load_pdf` and `chunk_document`?

In [13]:
# Your code here

**Conclusions:** *Your answer here.*